# Day 5 · Morning exercises (Exercise A)

**09:45–11:10 · then everyone back in the room at 11:10 for Peer review A**

Bayesian statistics for PhD researchers, Aalborg University 2026. Work in your group and keep your own copy of this notebook. Select **R (bayes)** and run the cells in order from a fresh kernel. Keep the folders `data/` and `models/` next to this notebook. The afternoon continues in `day-5-exercises-afternoon.ipynb`.

This morning nothing is fitted: you build the model and check what it says **before** it sees the data.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| A1 | Plot one person's attempts; choose a scale for x | 20 min |
| A2 | Priors and prior predictive simulation | 30 min |
| A3 | Fake data from the prior; read the Stan program | 20 min |
| — | Exit response and your group's question set | 15 min |

**Roles this block:** coder, checker, two question writers, interviewer, reporter. Rotate them in the afternoon.

**The data.** The Day 1 time game without feedback: each person tried ten times at each of the targets **5, 10 and 15 seconds**, with eyes closed. If the lecturer has placed the class's data as `data/timing-class.csv`, the notebook uses it; otherwise it uses `data/timing-backup.csv`, **synthetic** data for 15 fictional people. The column `elapsed_seconds` is the recorded time. Today we model **one person**: yourself, or the first person in the data.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
library(cmdstanr)    # Stan from R
set.seed(2026)
options(width = 95, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")
theme_set(theme_minimal(base_size = 14) + theme(legend.position = "top"))
DATA <- "data"; MODELS <- "models"     # folders next to this notebook

In [ ]:
# The class's Day 1 data if the lecturer has put them here, otherwise the synthetic backup
file <- if (file.exists(file.path(DATA, "timing-class.csv"))) "timing-class.csv" else "timing-backup.csv"
file
d_all <- read_csv(file.path(DATA, file), show_col_types = FALSE) |>
  filter(feedback_condition == "no_feedback", !is.na(elapsed_seconds))
d_all |> count(participant, target_seconds) |> head()

me <- d_all$participant[1]          # or your own ID from Day 1, for example "G3P07"
d <- d_all |>
  filter(participant == me) |>
  mutate(x = target_seconds - 10)   # centred predictor: 0 at the 10-s target
me
nrow(d)

## A1 (20 min): plot one person's attempts; choose a scale for x

**The model** (*slides "Does her clock scale?", "Regression: a model that generates the data" and "Choose a scale for x"*). Attempt i has target tᵢ and recorded time yᵢ. With the centred predictor xᵢ = tᵢ − 10,

yᵢ ~ Normal(α + β · xᵢ, σ), independent given x.

**Tasks.**

1. Plot recorded time against target, with the line of perfect timing (recorded = target).
2. Make a table with the number of attempts, the mean and the SD of the recorded times at each target.
3. In one sentence each, with units: what do α, β and σ mean on this scale? What would α mean with x = target instead of x = target − 10, and why is that a problem for choosing its prior?

In [ ]:
# Task 1
# ggplot(d, aes(x = target_seconds, y = elapsed_seconds)) +
#   geom_abline(intercept = 0, slope = 1, linetype = "dashed", colour = "grey50") +
#   geom_jitter(width = 0.25, height = 0, size = 3, alpha = 0.7, colour = "navy") +
#   labs(x = "target (s)", y = "recorded time (s)", caption = "dashed line: perfect timing")

# Task 2
# d |> group_by(target_seconds) |> summarise(attempts = n(), mean = mean(elapsed_seconds), sd = sd(elapsed_seconds))

**Your answer:** write a few sentences here.

## A2 (30 min): priors and prior predictive simulation

**The priors** (*slides "Priors on that scale", "Prior predictive simulation" and "What the prior predicts"*):

- α ~ Normal(10, 2): at the 10-s target she records about 10 s, probably 6–14 s;
- β ~ Normal(1, 0.5): recorded time grows with the target, roughly one-to-one;
- σ ~ Normal⁺(0, 2) (half-normal): attempts scatter by less than about 4 s.

A **prior predictive simulation** draws α, β and σ from their priors, computes the mean α + β · x at a target, and draws an attempt around it. No data are used.

**Tasks.**

1. Draw S = 4000 values of (α, β, σ) from the priors. For the targets 5, 10, 15 and 20 s, simulate one attempt per draw. Give the 90% interval of the simulated recorded time at each target and the share of negative times.
2. Plot 100 prior mean lines α + β · x for these priors and for a "vague" prior α ~ Normal(0, 20), β ~ Normal(0, 10), side by side, with the line of perfect timing.
3. For both priors, compute the share of draws with a **negative slope** β (recorded time falling as the target grows). Which prior expects impossible data?
4. Are the course priors plausible to you? If you would change one, say which and why, and rerun task 1 with it.

In [ ]:
# Task 1
S <- 4000
# prior <- tibble(alpha = rnorm(S, 10, 2),
#                 beta  = rnorm(S, 1, 0.5),
#                 sigma = abs(rnorm(S, 0, 2)))       # half-normal: keep the positive side
# prior_pred <- prior |>
#   cross_join(tibble(target = c(5, 10, 15, 20))) |>
#   mutate(y_sim = rnorm(n(), alpha + beta * (target - 10), sigma))
# prior_pred |> group_by(target) |>
#   summarise(q5 = quantile(y_sim, 0.05), q95 = quantile(y_sim, 0.95), share_negative = mean(y_sim < 0))

# Task 2: two tibbles of 100 draws each, stacked with bind_rows, then one row per target
# prior_lines <- bind_rows(
#     tibble(prior = "course prior", draw = 1:100, alpha = rnorm(100, 10, 2), beta = rnorm(100, 1, 0.5)),
#     tibble(prior = "vague prior",  draw = 1:100, alpha = rnorm(100, 0, 20), beta = rnorm(100, 0, 10))) |>
#   cross_join(tibble(target = seq(0, 25, by = 0.5))) |>
#   mutate(mean_line = alpha + beta * (target - 10))
# ggplot(prior_lines, aes(x = target, y = mean_line, group = draw)) + geom_line(alpha = 0.25, colour = "navy") +
#   geom_abline(intercept = 0, slope = 1, linetype = "dashed", colour = "tomato") + facet_wrap(~ prior) + ...

# Task 3: mean(prior$beta < 0);  mean(rnorm(S, 0, 10) < 0)

**Your answer:** write a few sentences here.

## A3 (20 min): fake data from the prior; read the Stan program

**The idea** (*slides "Fake data from the prior" and "The regression in Stan"*). A whole fake dataset uses **one** prior draw of (α, β, σ) for all its attempts, at the same targets as the real data.

**Tasks.**

1. Simulate six fake datasets at your person's targets and plot them in six panels with the line of perfect timing. Do they look like recorded times? Which features look wrong, if any?
2. Open `models/regression.stan` (in JupyterLab's file browser). For each line in the `model` block, say which part of the model above it is. What do `mu_20`, `y_20` and `y_rep` in `generated quantities` compute? (They are used this afternoon.)
3. Make the data list for Stan, with the **centred** x, and compile the model so that it is ready for the afternoon.

In [ ]:
# Task 1
# fake <- tibble(dataset = paste("fake dataset", 1:6),
#                alpha = rnorm(6, 10, 2), beta = rnorm(6, 1, 0.5), sigma = abs(rnorm(6, 0, 2))) |>
#   cross_join(d |> select(target_seconds, x)) |>
#   mutate(y_sim = rnorm(n(), alpha + beta * x, sigma))
# ggplot(fake, aes(x = target_seconds, y = y_sim)) + ... + facet_wrap(~ dataset, nrow = 2)

# Task 3
# stan_data <- list(N = nrow(d), x = d$x, y = d$elapsed_seconds)
# mod <- cmdstan_model(file.path(MODELS, "regression.stan"))

**Your answer:** write a few sentences here.

## Exit response

Write 80–120 words: What is a prior predictive simulation, and what does it check? Why does centring the predictor make it easier to choose priors?

**Your exit response:**

## Your group's question set

Question writers: write **three** questions about this block, each with your own worked solution:

1. a **concept** question (what does a result mean?),
2. a **computation** question (a number someone can check in R),
3. a **pitfall** question (a mistake that is easy to make).

The checker reviews the solutions. The reporter notes what your group found hardest, to ask in the plenary.

**Concept question:**

**Our solution:**

**Computation question:**

**Our solution:**

**Pitfall question:**

**Our solution:**

**Hardest open question (reporter):**

---
**Everyone back in the room at 11:10 for Peer review A.** Save this notebook (Ctrl/Cmd + S). After lunch, open `day-5-exercises-afternoon.ipynb`.